# LeetCode 3330 — Find the Original Typed String I bằng Python

Link đề: https://leetcode.com/problems/find-the-original-typed-string-i/ · Độ khó: **Easy** · Chủ đề: String, Counting, Run-length (nhóm ký tự liên tiếp)

Skeleton LeetCode:

```python
class Solution:
    def possibleStringCount(self, word: str) -> int:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Biweekly Contest 142); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: màn hình hiện chuỗi `word`; Alice **tối đa một lần** giữ phím quá lâu khiến một ký tự bị gõ lặp nhiều lần. Hỏi có bao nhiêu chuỗi **gốc** (chuỗi Alice định gõ) khác nhau có thể dẫn đến `word`.
2. Nắm các công cụ Python: duyệt chuỗi theo chỉ số, `zip(word, word[1:])` để lấy từng **cặp ký tự kề nhau**, `itertools.groupby` để tách **nhóm ký tự liên tiếp (run)**, `sum` trên giá trị `bool`, và `set` để loại trùng.
3. Đi từ cách **liệt kê mọi chuỗi gốc rồi đếm** → **đếm theo nhóm** `1 + Σ(độ dài nhóm − 1)` → **đếm một lượt các cặp kề nhau bằng nhau** → **`n − số lần đổi ký tự`** → **một dòng với `zip`**.
4. Hiểu mấu chốt: mỗi **cặp ký tự kề nhau giống nhau** ứng với **đúng một** chuỗi gốc khác (rút ngắn nhóm đó), nên đáp án là `1 + (số cặp kề nhau giống nhau)`.
5. Tránh các lỗi kinh điển: đếm số nhóm thay vì số cặp, chỉ đếm nhóm dài từ 2 trở lên, dùng `n - len(set(word))` (đếm cả ký tự trùng không liên tiếp), và so `word[i]` với `word[i-1]` khi `i = 0` (chỉ số `-1` quay vòng về cuối chuỗi).

## 2. Đề bài & ví dụ

Alice đang cố gõ một chuỗi nào đó trên máy tính. Cô hơi vụng: có thể cô **giữ phím quá lâu**, làm một ký tự bị gõ **nhiều lần**. Dù cố tập trung, cô biết mình có thể đã mắc lỗi này **tối đa một lần** (nghĩa là có thể một lần hoặc không lần nào).

Cho chuỗi `word` là **kết quả cuối cùng** hiện trên màn hình. Trả về **tổng số chuỗi gốc** (chuỗi Alice định gõ) có thể có.

**Ví dụ 1**

- Input: `word = "abbcccc"`
- Output: `5`
- Giải thích: các chuỗi gốc có thể là `"abbcccc"`, `"abbccc"`, `"abbcc"`, `"abbc"`, và `"abcccc"`.

**Ví dụ 2**

- Input: `word = "abcd"`
- Output: `1`
- Giải thích: chỉ có `"abcd"` (không có ký tự nào lặp, nên không thể có lỗi).

**Ví dụ 3**

- Input: `word = "aaaa"`
- Output: `4`
- Giải thích: gốc có thể là `"aaaa"`, `"aaa"`, `"aa"`, `"a"`.

**Ràng buộc:** `1 <= word.length <= 100`; `word` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
import random
import string
from itertools import groupby

examples = [
    ("abbcccc", 5),
    ("abcd", 1),
    ("aaaa", 4),
]
for word, expected in examples:
    print(repr(word), "-> kỳ vọng", expected)

'abbcccc' -> kỳ vọng 5
'abcd' -> kỳ vọng 1
'aaaa' -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. Chỉ số chuỗi, so sánh hai ký tự kề nhau

Chuỗi đánh chỉ số từ `0`. `word[i]` là ký tự thứ `i`, `word[i - 1]` là ký tự ngay trước nó. Hai ký tự kề nhau giống nhau khi `word[i] == word[i - 1]`. Chú ý: **chỉ số âm** quay vòng từ cuối chuỗi, `word[-1]` là ký tự **cuối cùng** (ta sẽ gặp bẫy này ở mục 13).

In [2]:
word = "abbc"
for i in range(1, len(word)):
    print(f"word[{i}]={word[i]!r}, word[{i - 1}]={word[i - 1]!r} -> bằng nhau? {word[i] == word[i - 1]}")
print(word[-1], word[0])                 # c a

word[1]='b', word[0]='a' -> bằng nhau? False
word[2]='b', word[1]='b' -> bằng nhau? True
word[3]='c', word[2]='b' -> bằng nhau? False
c a


### 3.2. "Nhóm" (run): các ký tự giống nhau **liên tiếp**

Một **run** là đoạn tối đa các ký tự giống nhau đứng cạnh nhau. Ví dụ `"abbcccc"` có 3 run: `a` (độ dài 1), `bb` (2), `cccc` (4). Ta có thể tách run bằng **hai con trỏ**: `i` là đầu run, `j` chạy tới khi gặp ký tự khác.

In [3]:
def split_runs(s):
    runs = []
    i = 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:    # đi tới hết run bắt đầu ở i
            j += 1
        runs.append((s[i], j - i))            # (ký tự, độ dài run)
        i = j
    return runs


print(split_runs("abbcccc"))             # [('a', 1), ('b', 2), ('c', 4)]
print(split_runs("aaaa"))                # [('a', 4)]
print(split_runs("abab"))                # [('a', 1), ('b', 1), ('a', 1), ('b', 1)]  (a, b không liên tiếp nên không gộp)

[('a', 1), ('b', 2), ('c', 4)]
[('a', 4)]
[('a', 1), ('b', 1), ('a', 1), ('b', 1)]


### 3.3. `itertools.groupby`: tách run có sẵn

`groupby(s)` gom các phần tử **liên tiếp bằng nhau**; mỗi lần lặp cho `(khoá, nhóm)` với `nhóm` là một iterator, ta đếm bằng `len(list(nhóm))` hoặc `sum(1 for _ in nhóm)`. Khác `Counter`, `groupby` **không gộp** các nhóm cách xa nhau.

In [4]:
for key, grp in groupby("aabccc"):
    print(key, len(list(grp)))           # a 2 / b 1 / c 3
print([(k, len(list(g))) for k, g in groupby("abab")])   # [('a', 1), ('b', 1), ('a', 1), ('b', 1)]

a 2
b 1
c 3
[('a', 1), ('b', 1), ('a', 1), ('b', 1)]


### 3.4. `zip(word, word[1:])`: lấy từng cặp ký tự kề nhau

`word[1:]` là chuỗi bỏ ký tự đầu. `zip(word, word[1:])` ghép phần tử cùng vị trí: `(word[0], word[1])`, `(word[1], word[2])`, ... và tự dừng khi hết chuỗi ngắn hơn. Đây chính là các cặp kề nhau (có `n - 1` cặp). Trong Python 3.10+ có thêm `itertools.pairwise(word)` làm đúng việc đó.

In [5]:
word = "abbc"
pairs = list(zip(word, word[1:]))
print(pairs)                             # [('a', 'b'), ('b', 'b'), ('b', 'c')]
print([a == b for a, b in pairs])        # [False, True, False]

[('a', 'b'), ('b', 'b'), ('b', 'c')]
[False, True, False]


### 3.5. `sum` trên giá trị `bool`

Trong Python `True` bằng `1` và `False` bằng `0` khi cộng. Nên `sum(điều_kiện for ...)` **đếm** số lần điều kiện đúng.

In [6]:
print(True + True, True + False)         # 2 1
print(sum([True, False, True]))          # 2
print(sum(c == "b" for c in "abbc"))     # 2   (đếm số chữ b)

2 1
2
2


### 3.6. `set` để loại trùng và ghép chuỗi

`set` là tập hợp không có phần tử trùng; `len(set(...))` đếm số phần tử **khác nhau**. Để dựng chuỗi mới ta cắt và nối: `s[:i] + "b" * m + s[j:]`.

In [7]:
cands = {"abc", "abc", "ab"}
print(cands, len(cands))                 # {'abc', 'ab'} 2 (thứ tự in có thể khác)
s = "abbcccc"
print(s[:1] + "b" * 1 + s[3:])           # abcccc  (rút nhóm bb còn một b)
print("b" * 2, "ab" + "c" * 3)           # bb abccc

{'ab', 'abc'} 2
abcccc
bb abccc


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Màn hình hiện `"abbcccc"`. Alice **tối đa một lần** giữ phím quá lâu. Có các khả năng:

1. **Không có lỗi:** chuỗi gốc chính là `"abbcccc"` (1 khả năng).
2. **Lỗi ở nhóm `bb`:** phím `b` bị giữ lâu; gốc có `b` ít hơn: chỉ còn **1 chữ b** (không thể là 0 chữ b — Alice đã gõ `b` ít nhất một lần). Gốc `"abcccc"`.
3. **Lỗi ở nhóm `cccc`:** gốc có `c` ít hơn: `3`, `2` hoặc `1` chữ `c`: `"abbccc"`, `"abbcc"`, `"abbc"`.
4. **Lỗi ở nhóm `a`** (độ dài 1): không thể rút gọn (không thể ít hơn 1) nên không thêm khả năng nào.

Tổng: `1 + 1 + 3 = 5`. Mỗi nhóm có độ dài `L` đóng góp thêm **`L − 1`** chuỗi gốc (độ dài mới `1, 2, ..., L − 1`).

In [8]:
word = "abbcccc"
runs = [(k, len(list(g))) for k, g in groupby(word)]
print(runs)                              # [('a', 1), ('b', 2), ('c', 4)]
extra = [(k, L - 1) for k, L in runs]
print("thêm mỗi nhóm:", extra)           # a:0, b:1, c:3
print("tổng =", 1 + sum(e for _, e in extra))   # 5

[('a', 1), ('b', 2), ('c', 4)]
thêm mỗi nhóm: [('a', 0), ('b', 1), ('c', 3)]
tổng = 5


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi chuỗi gốc rồi đếm

**Ý tưởng.** Làm đúng như mục 4: bắt đầu với tập `{word}` (không lỗi). Với mỗi nhóm bắt đầu ở `start`, độ dài `L`, và mỗi độ dài mới `new_len` từ `1` đến `L - 1`, tạo chuỗi gốc `word[:start] + ch * new_len + word[start + L:]` và thêm vào tập. Cuối cùng đếm `len(tập)`. Dùng `set` để chắc chắn không đếm trùng.

In [9]:
def possible_bruteforce(word: str) -> int:
    candidates = {word}                  # không lỗi: gốc = word
    i = 0
    n = len(word)
    while i < n:
        j = i
        while j < n and word[j] == word[i]:
            j += 1                       # [i, j) là một nhóm, độ dài L = j - i
        L = j - i
        for new_len in range(1, L):      # độ dài gốc 1..L-1 (khi L = 1 thì không có gì)
            candidates.add(word[:i] + word[i] * new_len + word[j:])
        i = j
    return len(candidates)


for word, expected in examples:
    print(possible_bruteforce(word), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [10]:
# xem tận mắt các chuỗi gốc của ví dụ 1
def list_originals(word):
    out = {word}
    i, n = 0, len(word)
    while i < n:
        j = i
        while j < n and word[j] == word[i]:
            j += 1
        for new_len in range(1, j - i):
            out.add(word[:i] + word[i] * new_len + word[j:])
        i = j
    return sorted(out)


print(list_originals("abbcccc"))         # ['abbc', 'abbcc', 'abbccc', 'abbcccc', 'abcccc']

['abbc', 'abbcc', 'abbccc', 'abbcccc', 'abcccc']


**Độ phức tạp (Cách 1).** Có tối đa `n - 1` chuỗi gốc phụ, mỗi chuỗi dựng tốn `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n²)`** (tập chứa tới `n` chuỗi dài `n`).

**Nút thắt:** ta dựng **cả chuỗi** chỉ để đếm. Số chuỗi phụ chỉ phụ thuộc **độ dài các nhóm**, không cần dựng từng chuỗi.

## 6. Ý tưởng thuật toán / bước đột phá

Mỗi nhóm độ dài `L` cho thêm đúng `L − 1` chuỗi gốc (độ dài mới `1..L−1`), và các chuỗi này **khác nhau** (khác nhóm hoặc khác độ dài) cũng **khác** `word` (vì có nhóm ngắn hơn). Do đó:

> đáp án = `1 + Σ (L − 1)` trên mọi nhóm.

Quan sát thêm: `Σ (L − 1)` chính là **số cặp ký tự kề nhau bằng nhau** `word[i] == word[i-1]`. Vì mỗi nhóm dài `L` có đúng `L − 1` cặp kề nhau bên trong, còn giữa hai nhóm khác nhau là hai ký tự khác nhau nên không tạo cặp bằng nhau. Do vậy:

> đáp án = `1 + (số vị trí i ≥ 1 mà word[i] == word[i-1])`.

Không cần tách nhóm, chỉ cần một lần quét.

In [11]:
for word, expected in examples + [("aabbbc", 4), ("abab", 1)]:
    runs = [len(list(g)) for _, g in groupby(word)]
    by_runs = 1 + sum(L - 1 for L in runs)
    by_pairs = 1 + sum(a == b for a, b in zip(word, word[1:]))
    print(f"{word!r:10} nhóm={runs} -> 1+Σ(L-1)={by_runs}, 1+số cặp bằng nhau={by_pairs}")
    assert by_runs == by_pairs

'abbcccc'  nhóm=[1, 2, 4] -> 1+Σ(L-1)=5, 1+số cặp bằng nhau=5
'abcd'     nhóm=[1, 1, 1, 1] -> 1+Σ(L-1)=1, 1+số cặp bằng nhau=1
'aaaa'     nhóm=[4] -> 1+Σ(L-1)=4, 1+số cặp bằng nhau=4
'aabbbc'   nhóm=[2, 3, 1] -> 1+Σ(L-1)=4, 1+số cặp bằng nhau=4
'abab'     nhóm=[1, 1, 1, 1] -> 1+Σ(L-1)=1, 1+số cặp bằng nhau=1


## 7. Các cách tiếp theo

### Cách 2 — Đếm theo nhóm bằng `groupby`

**Giải thích từng đoạn.**

1. `groupby(word)` trả từng nhóm ký tự liên tiếp bằng nhau.
2. `len(list(grp))` là độ dài `L` của nhóm.
3. Cộng `L - 1` của mọi nhóm, rồi `+ 1` cho trường hợp không có lỗi.

In [12]:
def possible_groupby(word: str) -> int:
    total = 1                            # không lỗi
    for _, grp in groupby(word):
        L = len(list(grp))               # độ dài nhóm
        total += L - 1                   # nhóm L đóng góp L - 1 chuỗi gốc phụ
    return total


for word, expected in examples:
    print(possible_groupby(word), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionGroupby:
    def possibleStringCount(self, word: str) -> int:
        from itertools import groupby
        return 1 + sum(len(list(g)) - 1 for _, g in groupby(word))


print(SolutionGroupby().possibleStringCount("abbcccc"))      # 5

5


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`** phụ (mỗi nhóm tạo list tạm dài `L`, tối đa `O(n)`).

### Cách 3 — Một lượt, đếm cặp kề nhau bằng nhau

**Giải thích từng đoạn.**

1. `count = 1` cho trường hợp không lỗi.
2. `for i in range(1, len(word))`: bắt đầu từ `1` để `word[i - 1]` luôn hợp lệ và **không** quay vòng.
3. Nếu `word[i] == word[i - 1]` thì `count += 1`.

In [14]:
def possible_one_pass(word: str) -> int:
    count = 1
    for i in range(1, len(word)):        # bắt đầu từ 1 để word[i - 1] hợp lệ
        if word[i] == word[i - 1]:
            count += 1
    return count


for word, expected in examples:
    print(possible_one_pass(word), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [15]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionOnePass:
    def possibleStringCount(self, word: str) -> int:
        count = 1
        for i in range(1, len(word)):
            if word[i] == word[i - 1]:
                count += 1
        return count


print(SolutionOnePass().possibleStringCount("aaaa"))         # 4

4


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 4 — `n − (số lần đổi ký tự)`

**Giải thích.** Có `n - 1` cặp kề nhau; mỗi cặp hoặc **bằng nhau** hoặc **khác nhau** ("đổi ký tự"). Vậy `số cặp bằng nhau = (n - 1) - số_lần_đổi`, suy ra đáp án `= 1 + (n - 1) - số_lần_đổi = n - số_lần_đổi`. Cách nhìn này hợp khi bạn đã đếm sẵn "số lần đổi" (ví dụ số nhóm `= số_lần_đổi + 1`). Ví dụ `"abbcccc"`: `n = 7`, đổi ở `a→b` và `b→c` ⇒ `2` lần ⇒ `7 - 2 = 5`.

In [16]:
def possible_by_changes(word: str) -> int:
    changes = 0
    for i in range(1, len(word)):
        if word[i] != word[i - 1]:       # đổi sang ký tự khác
            changes += 1
    return len(word) - changes


for word, expected in examples:
    print(possible_by_changes(word), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 5 — Một dòng với `zip` (hoặc `pairwise`)

**Giải thích.** Chính là Cách 3 viết gọn: `1 + sum(a == b for a, b in zip(word, word[1:]))`. Nếu Python của bạn từ 3.10 có thể viết `1 + sum(a == b for a, b in pairwise(word))` (`from itertools import pairwise`). Ta chọn `zip` vì chạy ở mọi phiên bản.

In [17]:
def possible_zip(word: str) -> int:
    return 1 + sum(a == b for a, b in zip(word, word[1:]))


for word, expected in examples:
    print(possible_zip(word), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
1 (kỳ vọng 1 )
4 (kỳ vọng 4 )


In [18]:
# Code nộp LeetCode cho Cách 5 (đổi tên lớp thành Solution khi nộp)
class SolutionZip:
    def possibleStringCount(self, word: str) -> int:
        return 1 + sum(a == b for a, b in zip(word, word[1:]))


print(SolutionZip().possibleStringCount("abbcccc"))          # 5

5


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (`word[1:]` tạo chuỗi mới; dùng `pairwise` thì `O(1)`).

## 8. TRACE từng bước

**Bảng trace Cách 3** với `word = "abbcccc"` (7 ký tự):

| i | `word[i-1]` | `word[i]` | bằng nhau? | `count` sau bước |
|---|---|---|---|---|
| (bắt đầu) | | | | 1 |
| 1 | `a` | `b` | không | 1 |
| 2 | `b` | `b` | **có** | 2 |
| 3 | `b` | `c` | không | 2 |
| 4 | `c` | `c` | **có** | 3 |
| 5 | `c` | `c` | **có** | 4 |
| 6 | `c` | `c` | **có** | 5 |

Kết quả `5`. Bốn dấu "có" tương ứng: `bb` cho 1 chuỗi (`abcccc`) và `cccc` cho 3 chuỗi.

In [19]:
def trace_one_pass(word: str) -> int:
    count = 1
    print(f"word = {word!r}, bắt đầu count = {count}")
    for i in range(1, len(word)):
        same = word[i] == word[i - 1]
        if same:
            count += 1
        print(f"  i={i}: word[{i - 1}]={word[i - 1]!r}, word[{i}]={word[i]!r}, bằng nhau? {same!s:5} -> count={count}")
    print("  kết quả:", count)
    return count


assert trace_one_pass("abbcccc") == 5
assert trace_one_pass("abcd") == 1

word = 'abbcccc', bắt đầu count = 1
  i=1: word[0]='a', word[1]='b', bằng nhau? False -> count=1
  i=2: word[1]='b', word[2]='b', bằng nhau? True  -> count=2
  i=3: word[2]='b', word[3]='c', bằng nhau? False -> count=2
  i=4: word[3]='c', word[4]='c', bằng nhau? True  -> count=3
  i=5: word[4]='c', word[5]='c', bằng nhau? True  -> count=4
  i=6: word[5]='c', word[6]='c', bằng nhau? True  -> count=5
  kết quả: 5
word = 'abcd', bắt đầu count = 1
  i=1: word[0]='a', word[1]='b', bằng nhau? False -> count=1
  i=2: word[1]='b', word[2]='c', bằng nhau? False -> count=1
  i=3: word[2]='c', word[3]='d', bằng nhau? False -> count=1
  kết quả: 1


In [20]:
def trace_groups(word: str) -> int:
    total = 1
    print(f"word = {word!r}, bắt đầu total = {total}")
    for key, grp in groupby(word):
        L = len(list(grp))
        total += L - 1
        print(f"  nhóm {key!r} độ dài {L}: cộng {L - 1} -> total={total}")
    print("  kết quả:", total)
    return total


assert trace_groups("abbcccc") == 5
assert trace_groups("aaaa") == 4

word = 'abbcccc', bắt đầu total = 1
  nhóm 'a' độ dài 1: cộng 0 -> total=1
  nhóm 'b' độ dài 2: cộng 1 -> total=2
  nhóm 'c' độ dài 4: cộng 3 -> total=5
  kết quả: 5
word = 'aaaa', bắt đầu total = 1
  nhóm 'a' độ dài 4: cộng 3 -> total=4
  kết quả: 4


## 9. Code nộp LeetCode

Ta nộp **Cách 3** (một lượt, đếm cặp kề nhau bằng nhau): ngắn, đúng, không cần thư viện.

In [21]:
class Solution:
    def possibleStringCount(self, word: str) -> int:
        count = 1                            # trường hợp không có lỗi
        for i in range(1, len(word)):        # bắt đầu từ 1 để word[i - 1] hợp lệ
            if word[i] == word[i - 1]:       # mỗi cặp kề nhau bằng nhau thêm 1 chuỗi gốc
                count += 1
        return count


sol = Solution()
print(sol.possibleStringCount("abbcccc"), sol.possibleStringCount("abcd"), sol.possibleStringCount("aaaa"))   # 5 1 4

5 1 4


## 10. Vì sao đúng

Gọi các nhóm của `word` có độ dài `L1, L2, ..., Lm` (với `L1 + ... + Lm = n`).

- **Mọi chuỗi gốc hợp lệ có dạng nào?** Alice mắc lỗi tối đa một lần, tức **tối đa một lần nhấn** bị giữ lâu. Nếu không lỗi: gốc `= word`. Nếu có lỗi: gốc có một nhóm ký tự `ch` độ dài `l ≥ 1` và một ký tự trong nhóm đó bị giữ lâu thành `m ≥ 2` ký tự, nên nhóm trên màn hình dài `Lt = (l − 1) + m ≥ l + 1`. Ngược lại, với mọi `l` từ `1` đến `Lt − 1` ta chọn `m = Lt − l + 1 ≥ 2` là được. Các nhóm khác giữ nguyên. Vậy nhóm `t` cho đúng `Lt − 1` chuỗi gốc (và `0` nếu `Lt = 1`).
- **Các chuỗi đó đôi một khác nhau:** hai chuỗi từ hai nhóm khác nhau khác nhau ở độ dài đúng một nhóm khác nhau; hai chuỗi cùng nhóm nhưng khác độ dài gốc khác nhau rõ ràng; còn mọi chuỗi phụ đều ngắn hơn `word` nên khác `word`.
- Vậy đáp án `= 1 + Σ (Lt − 1) = 1 + (n − m)`. Mà `n − m` là số cặp kề nhau bằng nhau (mỗi nhóm `Lt` có `Lt − 1` cặp, giữa hai nhóm là hai ký tự khác nhau). Đó chính là Cách 3.

Ta cũng kiểm chứng bằng một **bộ đếm độc lập** ở mục 12: duyệt **mọi chuỗi gốc có thể** (trên bảng chữ cái nhỏ) và kiểm tra xem nó có thể sinh ra `word` bằng "tối đa một lần giữ phím lâu" hay không.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê chuỗi gốc | `O(n²)` | `O(n²)` |
| 2. `groupby` | `O(n)` | `O(1)` phụ |
| 3. Một lượt, đếm cặp bằng nhau | `O(n)` | `O(1)` |
| 4. `n − số_lần_đổi` | `O(n)` | `O(1)` |
| 5. Một dòng `zip` | `O(n)` | `O(n)` (`word[1:]`) |

Mọi cách tuyến tính đều đủ nhanh; việc quan trọng là **tìm ra công thức** thay vì liệt kê.

## 12. Bộ test

Để đối chiếu **độc lập**, ta viết `reference_by_generation(word)`: duyệt **mọi chuỗi gốc** có độ dài từ `1` đến `len(word)` trên bảng chữ cái gồm các chữ có mặt trong `word` (dùng cho `word` nhỏ), và đếm những chuỗi có thể sinh ra `word` bằng "tối đa một lần giữ phím lâu". Hàm `can_produce(orig, word)` mô phỏng đề: `orig == word`, hoặc tồn tại vị trí `i` sao cho ký tự `orig[i]` bị gõ lặp `m ≥ 2` lần: `orig[:i] + orig[i] * m + orig[i+1:] == word`.

In [22]:
from itertools import product


def can_produce(orig, word):
    if orig == word:
        return True
    for i in range(len(orig)):
        for m in range(2, len(word) + 1):
            if orig[:i] + orig[i] * m + orig[i + 1:] == word:
                return True
    return False


def reference_by_generation(word):
    letters = sorted(set(word))
    count = 0
    for length in range(1, len(word) + 1):
        for tup in product(letters, repeat=length):
            if can_produce("".join(tup), word):
                count += 1
    return count


impls = {
    "bruteforce": possible_bruteforce,
    "groupby": possible_groupby,
    "one_pass": possible_one_pass,
    "by_changes": possible_by_changes,
    "zip": possible_zip,
    "leetcode": Solution().possibleStringCount,
}


def check_all(word, expected, label):
    for name, fn in impls.items():
        actual = fn(word)
        assert actual == expected, (label, name, word, actual, expected)
    print("PASS", label)


for idx, (word, expected) in enumerate(examples, 1):
    check_all(word, expected, f"ví dụ {idx}")
    assert reference_by_generation(word) == expected
print("bộ đếm độc lập khớp 3 ví dụ")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3
bộ đếm độc lập khớp 3 ví dụ


### 12.1. Các trường hợp biên

In [23]:
check_all("a", 1, "một ký tự")
check_all("aa", 2, "hai ký tự giống nhau")
check_all("ab", 1, "hai ký tự khác nhau")
check_all("aab", 2, "nhóm đầu dài 2")
check_all("abb", 2, "nhóm cuối dài 2")
check_all("aabb", 3, "hai nhóm dài 2")
check_all("abab", 1, "ký tự trùng nhưng không liên tiếp")
check_all("aba", 1, "đầu và cuối giống nhau (bẫy chỉ số -1)")
check_all("a" * 100, 100, "100 chữ a")
check_all("ab" * 50, 1, "xen kẽ độ dài 100")
check_all("a" * 50 + "b" * 50, 99, "hai nhóm dài 50")
check_all("abcdefghijklmnopqrstuvwxyz", 1, "26 chữ khác nhau")
check_all("aabbbc", 4, "ví dụ tự dựng ở bài tập 2")
for w in ["a", "aa", "ab", "aab", "abb", "aabb", "abab", "aba", "aabbbc"]:
    assert reference_by_generation(w) == Solution().possibleStringCount(w), w
print("PASS bộ đếm độc lập khớp các biên nhỏ")

PASS một ký tự
PASS hai ký tự giống nhau
PASS hai ký tự khác nhau
PASS nhóm đầu dài 2
PASS nhóm cuối dài 2
PASS hai nhóm dài 2
PASS ký tự trùng nhưng không liên tiếp
PASS đầu và cuối giống nhau (bẫy chỉ số -1)
PASS 100 chữ a
PASS xen kẽ độ dài 100
PASS hai nhóm dài 50
PASS 26 chữ khác nhau
PASS ví dụ tự dựng ở bài tập 2
PASS bộ đếm độc lập khớp các biên nhỏ


### 12.2. Phản ví dụ cho các cách làm sai

Bốn lỗi: (a) đếm số nhóm; (b) đếm số nhóm có độ dài `>= 2`; (c) `n - len(set(word))`; (d) `range(len(word))` làm `word[-1]` quay vòng. Mỗi lỗi sai ở một ví dụ cụ thể.

In [24]:
def wrong_count_groups(word):
    return len([1 for _ in groupby(word)])                  # SAI: số nhóm, không phải số cặp


def wrong_only_long_groups(word):
    return 1 + sum(1 for _, g in groupby(word) if len(list(g)) >= 2)   # SAI: nhóm dài L cho L-1 chuỗi, không phải 1


def wrong_set(word):
    return len(word) - len(set(word)) + 1                   # SAI: đếm cả ký tự trùng không liên tiếp


def wrong_wrap(word):
    return 1 + sum(word[i] == word[i - 1] for i in range(len(word)))   # SAI: i = 0 so với word[-1]


tests = [("abbcccc", 5), ("abab", 1), ("aba", 1)]
for fn in (wrong_count_groups, wrong_only_long_groups, wrong_set, wrong_wrap):
    results = [(w, fn(w), exp) for w, exp in tests]
    print(fn.__name__, results)
    assert any(got != exp for _, got, exp in results), fn.__name__

wrong_count_groups [('abbcccc', 3, 5), ('abab', 4, 1), ('aba', 3, 1)]
wrong_only_long_groups [('abbcccc', 3, 5), ('abab', 1, 1), ('aba', 1, 1)]
wrong_set [('abbcccc', 5, 5), ('abab', 3, 1), ('aba', 2, 1)]
wrong_wrap [('abbcccc', 5, 5), ('abab', 1, 1), ('aba', 2, 1)]


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

(1) Với chuỗi **nhỏ** (độ dài `1..7`, bảng chữ cái `1..3` chữ): so mọi cách với `reference_by_generation` (bộ đếm độc lập). (2) Với chuỗi **lớn** (độ dài `1..100`, chữ thường, đa dạng cỡ bảng chữ cái để có cả nhóm dài lẫn không): so các cách với nhau và với `1 + Σ(L − 1)` tính trực tiếp.

In [25]:
rng = random.Random(3330)

# (1) chuỗi nhỏ, đối chiếu với bộ đếm độc lập
checked = 0
for _ in range(400):
    n = rng.randint(1, 7)
    alpha = rng.choice(["a", "ab", "abc"])
    word = "".join(rng.choice(alpha) for _ in range(n))
    ref = reference_by_generation(word)
    for name, fn in impls.items():
        assert fn(word) == ref, (name, word, fn(word), ref)
    checked += 1
print("PASS", checked, "chuỗi nhỏ so với bộ đếm độc lập")

# (2) chuỗi lớn trong ràng buộc
checked = 0
for _ in range(2000):
    n = rng.randint(1, 100)
    alpha = rng.choice([2, 3, 5, 26])
    word = "".join(rng.choice(string.ascii_lowercase[:alpha]) for _ in range(n))
    assert 1 <= len(word) <= 100 and word.islower()
    ref = 1 + sum(len(list(g)) - 1 for _, g in groupby(word))
    for name, fn in impls.items():
        assert fn(word) == ref, (name, word)
    checked += 1
print("PASS", checked, "chuỗi lớn ngẫu nhiên")

PASS 400 chuỗi nhỏ so với bộ đếm độc lập
PASS 2000 chuỗi lớn ngẫu nhiên


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số nhóm thay vì số cặp bằng nhau

In [26]:
def sai_dem_nhom(word):
    groups = 1
    for i in range(1, len(word)):
        if word[i] != word[i - 1]:       # SAI: đếm số lần đổi ký tự = số nhóm
            groups += 1
    return groups


print("sai:", sai_dem_nhom("abbcccc"), "| đúng:", Solution().possibleStringCount("abbcccc"))   # 3 | 5
print("sai:", sai_dem_nhom("abcd"), "| đúng:", Solution().possibleStringCount("abcd"))         # 4 | 1

sai: 3 | đúng: 5
sai: 4 | đúng: 1


**Vì sao sai:** số nhóm bằng `1 + số_lần_đổi`, **ngược** với điều ta cần (số cặp bằng nhau). Với `"abcd"` có 4 nhóm nhưng chỉ 1 chuỗi gốc. **Cách sửa:** đếm các cặp **bằng nhau** (`==`), hoặc dùng `n − số_lần_đổi`.

### Lỗi 2: Mỗi nhóm dài `>= 2` chỉ tính là 1 khả năng

In [27]:
def sai_moi_nhom_mot(word):
    return 1 + sum(1 for _, g in groupby(word) if len(list(g)) >= 2)


print("sai:", sai_moi_nhom_mot("aaaa"), "| đúng:", Solution().possibleStringCount("aaaa"))    # 2 | 4
print("sai:", sai_moi_nhom_mot("abbcccc"), "| đúng:", Solution().possibleStringCount("abbcccc"))  # 3 | 5

sai: 2 | đúng: 4
sai: 3 | đúng: 5


**Vì sao sai:** gốc có thể rút gọn nhóm tới **bất kỳ** độ dài từ `1` đến `L − 1`, tức `L − 1` khả năng, không phải 1. Với `"aaaa"` ta có `"aaa"`, `"aa"`, `"a"` (3 khả năng) cộng `"aaaa"` là 4. **Cách sửa:** cộng `L − 1` (hoặc đếm từng cặp bằng nhau, mỗi cặp là một đơn vị).

### Lỗi 3: Dùng `n − len(set(word))` (đếm cả ký tự trùng không liên tiếp)

In [28]:
def sai_dung_set(word):
    return len(word) - len(set(word)) + 1


print("sai:", sai_dung_set("abab"), "| đúng:", Solution().possibleStringCount("abab"))   # 3 | 1
print("sai:", sai_dung_set("abba"), "| đúng:", Solution().possibleStringCount("abba"))   # 3 | 2

sai: 3 | đúng: 1
sai: 3 | đúng: 2


**Vì sao sai:** `set` đếm số ký tự **khác nhau** trên cả chuỗi, bỏ qua chuyện chúng có **kề nhau** hay không. `"abab"` có 2 chữ `a` nhưng không kề nhau, nên không thể là "giữ phím lâu" ⇒ đáp án là 1. Đề chỉ quan tâm các lần lặp **liên tiếp**. **Cách sửa:** so sánh **hai ký tự đứng cạnh nhau** (`word[i] == word[i - 1]`).

### Lỗi 4: Chỉ số `-1` quay vòng khi bắt đầu từ `i = 0`

In [29]:
def sai_chi_so_am(word):
    count = 1
    for i in range(len(word)):           # SAI: i = 0 -> word[i - 1] = word[-1] (ký tự CUỐI)
        if word[i] == word[i - 1]:
            count += 1
    return count


print("sai:", sai_chi_so_am("aba"), "| đúng:", Solution().possibleStringCount("aba"))     # 2 | 1
print("sai:", sai_chi_so_am("a"), "| đúng:", Solution().possibleStringCount("a"))         # 2 | 1

sai: 2 | đúng: 1
sai: 2 | đúng: 1


**Vì sao sai:** Python cho phép chỉ số âm, `word[-1]` là ký tự cuối. Khi `i = 0`, phép so `word[0] == word[-1]` so **đầu chuỗi với cuối chuỗi**, thêm một lần đếm "ma" nếu chúng giống nhau (`"aba"`, hoặc chuỗi một ký tự `"a"` tự so với chính nó). **Cách sửa:** bắt đầu vòng lặp từ `1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ký tự **lặp liên tiếp**" / "đoạn tối đa các ký tự giống nhau" ⇒ nghĩ ngay đến **run** (`groupby`, hai con trỏ) hoặc so sánh **cặp kề nhau**.
- Khi phải đếm số khả năng, đừng vội liệt kê: thử vài ví dụ, tìm công thức (ở đây `1 + Σ(L − 1)`), rồi **kiểm chứng bằng vét cạn** trên input nhỏ.
- `Σ(L − 1)` trên các nhóm = `n − số nhóm` = số cặp kề nhau bằng nhau: ba cách nhìn của cùng một đại lượng; chọn cách dễ code nhất.
- Duyệt cặp kề nhau: `for i in range(1, n)` (dùng `word[i-1]`), hoặc `zip(word, word[1:])`, hoặc `pairwise`. Không bắt đầu từ `0` khi dùng `i - 1`.
- `Counter` hoặc `set` **không** biết thứ tự/vị trí kề nhau; khi đề nói "liên tiếp" đừng dùng chúng thay cho so sánh kề nhau.

## 15. Biến thể của bài

1. **Find the Original Typed String II (3333, Hard):** thêm ràng buộc độ dài gốc **ít nhất `k`**, cần quy hoạch động / đếm có tổ hợp. (Chi tiết đề: đối chiếu lại đề gốc.)
2. **Consecutive Characters (1446):** độ dài **nhóm dài nhất**.
3. **String Compression (443)**, **Count and Say (38)**: làm việc với run-length.
4. **Lỗi bất kỳ số lần** (mỗi nhóm có thể mắc lỗi độc lập): mỗi nhóm có `L` lựa chọn độ dài gốc `1..L` ⇒ tích `Π L`.
5. **Phải có đúng một lỗi** (không tính chuỗi `word` gốc): đáp án `Σ(L − 1)` = đáp án bài này trừ 1.

In [30]:
def count_any_runs(word):
    # mỗi nhóm độc lập chọn độ dài gốc 1..L  =>  tích các L
    result = 1
    for _, g in groupby(word):
        result *= len(list(g))
    return result


def count_exactly_one_mistake(word):
    return Solution().possibleStringCount(word) - 1


def longest_run(word):
    return max(len(list(g)) for _, g in groupby(word))


def brute_any_runs(word):
    # kiểm chứng: dựng mọi tổ hợp độ dài gốc cho từng nhóm
    runs = [(k, len(list(g))) for k, g in groupby(word)]
    originals = {""}
    for ch, L in runs:
        originals = {o + ch * m for o in originals for m in range(1, L + 1)}
    return len(originals)


for w in ["abbcccc", "abcd", "aaaa", "aabbb"]:
    assert count_any_runs(w) == brute_any_runs(w), w
print(count_any_runs("abbcccc"), count_exactly_one_mistake("abbcccc"), longest_run("abbcccc"))   # 8 4 4
assert count_any_runs("abbcccc") == 8 and count_exactly_one_mistake("abbcccc") == 4 and longest_run("abbcccc") == 4

8 4 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê chuỗi gốc | dựng từng chuỗi gốc vào `set` | `O(n²)` | `O(n²)` | hiểu đề, đối chiếu |
| 2. `groupby` | `1 + Σ(L − 1)` | `O(n)` | `O(1)` phụ | khi cần độ dài từng nhóm |
| 3. Một lượt | đếm cặp kề nhau bằng nhau | `O(n)` | `O(1)` | **nên nộp** |
| 4. `n − số_lần_đổi` | đếm cặp khác nhau rồi trừ | `O(n)` | `O(1)` | đã có sẵn số lần đổi / số nhóm |
| 5. Một dòng `zip` | Cách 3 viết gọn | `O(n)` | `O(n)` | code ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"aab"[0] == "aab"[1]`, `list(zip("abc", "bc"))`, `sum([True, False, True])`, `Solution().possibleStringCount("aabb")`, `Solution().possibleStringCount("aaa")`.

**Bài 2 (tính tay).** Với `word = "aabbbc"`: tách nhóm, tính đáp án, rồi **liệt kê** đủ các chuỗi gốc.

**Bài 3 (điền TODO).** Hoàn thành `count_adjacent_equal(word)` — đếm số cặp kề nhau bằng nhau.

In [31]:
def count_adjacent_equal(word):
    # TODO 1: count = 0
    # TODO 2: for i in range(1, len(word)): nếu word[i] == word[i - 1] thì count += 1
    # TODO 3: return count
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `possible_runs(word)` theo Cách 2 bằng **hai con trỏ** (không dùng `groupby`): `i` là đầu nhóm, `j` chạy tới hết nhóm, cộng `j - i - 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_any_runs(word)` = số chuỗi gốc nếu **mỗi nhóm có thể mắc lỗi độc lập** (tích các độ dài nhóm). (b) Viết `list_originals_at_most_once(word)` trả về **danh sách tăng dần theo thứ tự từ điển** các chuỗi gốc của bài này, rồi kiểm tra `len(...)` bằng `possibleStringCount`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `[('a', 'b'), ('b', 'c')]`, `2`, `3`, `3`. Bài 3: vòng `for i in range(1, len(word))`, so với `word[i - 1]`.
- **Mức 2 (vừa):** Bài 2: các nhóm `aa`, `bbb`, `c` ⇒ `1 + 1 + 2 + 0 = 4`; các chuỗi gốc: `aabbbc`, `abbbc`, `aabbc`, `aabc`. Bài 4: sau vòng `while j < n and word[j] == word[i]`, độ dài nhóm là `j - i`, rồi đặt `i = j`.
- **Mức 3 (sâu):** Bài 5(a): tích độ dài từng nhóm (khởi tạo `1`). Bài 5(b): giống Cách 1 (tập `set`), cuối cùng `sorted(...)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert ("aab"[0] == "aab"[1]) is True
assert list(zip("abc", "bc")) == [("a", "b"), ("b", "c")]
assert sum([True, False, True]) == 2
assert Solution().possibleStringCount("aabb") == 3
assert Solution().possibleStringCount("aaa") == 3

# Bài 2
assert Solution().possibleStringCount("aabbbc") == 4
assert list_originals("aabbbc") == sorted(["aabbbc", "abbbc", "aabbc", "aabc"])


# Bài 3
def count_adjacent_equal_ans(word):
    count = 0
    for i in range(1, len(word)):
        if word[i] == word[i - 1]:
            count += 1
    return count


for word, expected in examples:
    assert 1 + count_adjacent_equal_ans(word) == expected


# Bài 4
def possible_runs(word):
    total = 1
    i, n = 0, len(word)
    while i < n:
        j = i
        while j < n and word[j] == word[i]:
            j += 1
        total += j - i - 1
        i = j
    return total


for word, expected in examples:
    assert possible_runs(word) == expected


# Bài 5(a)
def count_any_runs_ans(word):
    result = 1
    i, n = 0, len(word)
    while i < n:
        j = i
        while j < n and word[j] == word[i]:
            j += 1
        result *= j - i
        i = j
    return result


assert count_any_runs_ans("abbcccc") == 8
assert count_any_runs_ans("aaaa") == 4
assert all(count_any_runs_ans(w) == brute_any_runs(w) for w in ["abb", "aabb", "abcabc", "aaabbbccc"])


# Bài 5(b)
def list_originals_at_most_once(word):
    out = {word}
    i, n = 0, len(word)
    while i < n:
        j = i
        while j < n and word[j] == word[i]:
            j += 1
        for m in range(1, j - i):
            out.add(word[:i] + word[i] * m + word[j:])
        i = j
    return sorted(out)


for word, expected in examples:
    lst = list_originals_at_most_once(word)
    assert len(lst) == expected == Solution().possibleStringCount(word)
    assert lst == sorted(lst)
assert list_originals_at_most_once("abbcccc") == ["abbc", "abbcc", "abbccc", "abbcccc", "abcccc"]
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm cặp kề nhau bằng nhau)

```python
class Solution:
    def possibleStringCount(self, word: str) -> int:
        count = 1
        for i in range(1, len(word)):
            if word[i] == word[i - 1]:
                count += 1
        return count
```

**Checklist trước khi submit**

- [ ] Có `+ 1` cho trường hợp "không có lỗi" (chuỗi gốc chính là `word`)?
- [ ] Đếm cặp **kề nhau bằng nhau** (không phải số nhóm, không phải `n - len(set)`)?
- [ ] Vòng lặp bắt đầu từ `i = 1` (tránh `word[-1]`)?
- [ ] Nhóm dài `L` đóng góp `L − 1` (không phải 1)?
- [ ] Đã thử `"abbcccc" -> 5`, `"abcd" -> 1`, `"aaaa" -> 4`, `"a" -> 1`, `"aba" -> 1`?
- [ ] Hàm trả về số nguyên (`int`), không phải danh sách chuỗi?